# Calcium Transient Analysis Pipeline

Generalized pipeline for ND2 calcium-imaging experiments on cardiac microbundles.

**Workflow — run cells top to bottom:**

1. **STEP 0** — Environment (imports + core analysis function)
2. **STEP 1** — Configure the experiment (edit `BASE`, `PAIR_MAP`, timepoint labels)
3. **STEP 2** — Discover files, build tissue-pair index
4. **STEP 3** — Contact sheet: auto-ROI QC (visually check tissue segmentation)
5. **STEP 4** — Optional: manual ROI picker for any bad auto-detections
6. **STEP 5** — Batch analysis (cached, resumable)
7. **STEP 6** — Trace-picker QC (click bad traces to exclude)
8. **STEP 7** — Stats + figures (paired within-line, between-line, Δ interaction)
9. **STEP 8** — Snapshot everything for archival

**Data layout expected:** `<BASE>/<LINE>/<TIMEPOINT_SUBFOLDER>/*.nd2`

**Persistent files** (in your home dir, one per experiment machine):
- `~/calcium_manual_rois.json` — manual ROI overrides
- `~/calcium_qc_exclude.json` — excluded basenames
- `~/calcium_runs_cache/` — per-file pickle cache (skip re-analysis)

Adapted from the MINT vs GEM12 analysis (see snapshot at `~/calcium_snapshots/`).

## STEP 0 — Environment

In [ ]:
# One-time package install (skip if already installed)
# %pip install nd2 pandas scikit-image ipywidgets

In [ ]:
import os
import time
import nd2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, peak_widths, savgol_filter
from scipy.ndimage import distance_transform_edt, binary_fill_holes
from scipy.optimize import curve_fit
from scipy import stats as sps
from skimage.filters import threshold_li, threshold_otsu, threshold_triangle, gaussian
from skimage.morphology import closing, opening, disk, convex_hull_image
from skimage.measure import label, regionprops
print("Environment loaded.")

In [ ]:
# ============================================================
# CORE ANALYSIS FUNCTION  —  process_and_verify_roi(...)
# ============================================================
# Detects tissue in a rectangular microbundle ND2 recording, extracts
# background-subtracted dF/F0, detects peaks, and returns metrics
# per tissue: beat rate, amplitude, rise time, CaTD50, CaTD90, decay τ.
# Accepts optional tissue_roi=(y1,y2,x1,x2) to override auto-detection.
# ============================================================

def _fit_decay_tau(trace, peak_idx, fps, next_peak_idx=None,
                   min_span_ms=30, max_span_ms=600):
    n = len(trace); start = int(peak_idx)
    hard_end = start + int((max_span_ms / 1000.0) * fps)
    if next_peak_idx is not None:
        hard_end = min(hard_end, int((peak_idx + next_peak_idx) / 2))
    hard_end = min(hard_end, n - 1)
    min_pts = max(4, int((min_span_ms / 1000.0) * fps))
    if hard_end - start < min_pts: return float("nan")
    t = np.arange(hard_end - start + 1) / fps
    y = trace[start:hard_end + 1].astype(float)
    A0 = max(y[0] - y[-1], 1e-6); tau0 = 0.1; C0 = y[-1]
    try:
        popt, _ = curve_fit(lambda x, A, tau, C: A * np.exp(-x / tau) + C,
                            t, y, p0=(A0, tau0, C0), maxfev=2000,
                            bounds=([0, 1e-3, -np.inf], [np.inf, 5.0, np.inf]))
        return popt[1] * 1000.0
    except Exception:
        return float("nan")

def _robust_baseline_noise(x):
    """MAD of first differences / sqrt(2). Invariant to transient shape."""
    x = np.asarray(x, dtype=float)
    if x.size < 4: return 0.0
    d = np.diff(x); med = np.median(d)
    mad = 1.4826 * float(np.median(np.abs(d - med)))
    return mad / np.sqrt(2.0)

def _rise_time_10_90(trace, peak_idx, fps, prev_peak_idx=None,
                     search_span_ms=400):
    n = len(trace); peak_idx = int(peak_idx)
    if peak_idx <= 0: return float("nan")
    ws = max(0, peak_idx - int((search_span_ms / 1000.0) * fps))
    if prev_peak_idx is not None:
        ws = max(ws, int(prev_peak_idx) + 1)
    if ws >= peak_idx: return float("nan")
    up = trace[ws:peak_idx + 1].astype(float)
    b = float(up.min()); pv = float(trace[peak_idx]); amp = pv - b
    if amp <= 0: return float("nan")
    def _cross(y, thr):
        ab = y >= thr
        if not ab.any() or ab.all(): return None
        idxs = np.where(ab[1:] & ~ab[:-1])[0]
        if len(idxs) == 0: return None
        i = int(idxs[-1]); y0, y1 = y[i], y[i + 1]
        if y1 == y0: return float(i + 1)
        return i + (thr - y0) / (y1 - y0)
    i10 = _cross(up, b + 0.1 * amp)
    i90 = _cross(up, b + 0.9 * amp)
    if i10 is None or i90 is None or i90 <= i10: return float("nan")
    return (i90 - i10) / fps * 1000.0

def _upstroke_rate(trace, peaks, fps, search_span_ms=400):
    if len(peaks) == 0: return float("nan")
    span = max(3, int(round((search_span_ms / 1000.0) * fps)))
    rates = []
    for p in peaks:
        p = int(p); lo = max(0, p - span)
        seg = trace[lo:p + 1]
        if seg.size < 2: continue
        d = np.diff(seg) * fps
        rmax = float(d.max()) if d.size else float("nan")
        if np.isfinite(rmax) and rmax > 0: rates.append(rmax)
    return float(np.median(rates)) if rates else float("nan")

def _iri(trace, peaks, fps, guard_ms=40):
    trace = np.asarray(trace, dtype=float); peaks = np.asarray(peaks, dtype=int)
    if peaks.size < 2: return float("nan")
    guard = max(1, int(round((guard_ms / 1000.0) * fps)))
    rec_min = float(np.nanmin(trace))
    iris = []
    for i in range(peaks.size - 1):
        p, q = int(peaks[i]), int(peaks[i + 1])
        lo = p + guard; hi = q - guard
        if hi - lo < 2: continue
        trough = float(np.nanmin(trace[lo:hi]))
        amp = float(trace[p]) - rec_min
        if amp <= 0: continue
        val = (trough - rec_min) / amp
        if np.isfinite(val): iris.append(val)
    return float(np.median(iris)) if iris else float("nan")

def _read_fps_from_nd2(nd_file):
    try:
        for lvl in nd_file.experiment or []:
            params = getattr(lvl, "parameters", None)
            periods = getattr(params, "periods", None) if params else None
            if periods and len(periods) > 0:
                per = periods[0]; pd_ = getattr(per, "periodDiff", None)
                avg_ms = getattr(pd_, "avg", None) if pd_ else None
                if avg_ms and avg_ms > 0: return 1000.0 / float(avg_ms)
    except Exception:
        pass
    try:
        events = nd_file.events()
        if events and "Time [s]" in events[0]:
            ts = np.array([e["Time [s]"] for e in events])
            if len(ts) >= 2:
                dt = np.median(np.diff(ts))
                if dt > 0: return 1.0 / dt
    except Exception:
        pass
    return None

def _smooth_trace(y, fps, target_ms=40.0, polyorder=1):
    n = len(y); win = int(round((target_ms / 1000.0) * fps))
    if win % 2 == 0: win += 1
    win = max(win, polyorder + 3)
    if win % 2 == 0: win += 1
    if n < win or win < 5: return np.asarray(y, dtype=float), 0
    return savgol_filter(y, window_length=win, polyorder=polyorder,
                         mode="interp"), win

def _score_candidate(mask, H, W, expected_area_frac,
                      area_frac_range, aspect_range, must_contain_yx):
    area = float(mask.sum()); total = H * W
    if area == 0: return None
    frac = area / total
    if frac < area_frac_range[0] or frac > area_frac_range[1]: return None
    y, x = must_contain_yx
    if not (0 <= y < H and 0 <= x < W and mask[y, x]): return None
    edge_hits = int(mask[0].any()) + int(mask[-1].any()) + \
                int(mask[:, 0].any()) + int(mask[:, -1].any())
    if edge_hits >= 3: return None
    ys, xs = np.where(mask)
    h = ys.max() - ys.min() + 1; w = xs.max() - xs.min() + 1
    long_ax, short_ax = max(h, w), max(1, min(h, w))
    aspect = long_ax / short_ax
    if aspect < aspect_range[0] or aspect > aspect_range[1]: return None
    bbox_area = h * w; fill = area / bbox_area
    ln_ratio = np.log(frac / expected_area_frac)
    area_score = float(np.exp(-0.5 * (ln_ratio / 0.7) ** 2))
    aspect_score = float(min(1.0, (aspect - 1.0) / 4.0))
    border = int(round(0.10 * min(H, W)))
    if border > 0:
        interior = mask.copy()
        interior[:border] = False; interior[-border:] = False
        interior[:, :border] = False; interior[:, -border:] = False
        centered = float(interior.sum() / area)
    else:
        centered = 1.0
    fill_s = float(min(1.0, max(0.0, (fill - 0.3) / 0.5)))
    return 0.4 * area_score + 0.15 * aspect_score + 0.2 * centered + 0.25 * fill_s

def _detect_tissue_mask(video_f32, activity_map,
                        closing_disk_px=15, opening_disk_px=5,
                        use_convex_hull=True, expected_area_frac=0.30,
                        area_frac_range=(0.05, 0.75),
                        aspect_range=(1.5, 15.0), verbose=False):
    H, W = activity_map.shape
    act_smooth = gaussian(activity_map, sigma=6)
    anchor = int(np.argmax(act_smooth))
    anchor_yx = (anchor // W, anchor % W)
    mean_frame = video_f32.mean(axis=0)
    candidates = []
    def _build(name, raw, open_r, close_r):
        if raw.sum() < 100: return
        m = opening(raw, disk(open_r)) if open_r > 0 else raw.copy()
        if m.sum() < 100: return
        m = closing(m, disk(close_r))
        lab = label(m); regs = regionprops(lab)
        if not regs: return
        ay, ax = anchor_yx
        anchor_label = int(lab[ay, ax])
        if anchor_label > 0:
            m1 = (lab == anchor_label)
        else:
            regs.sort(key=lambda r: r.area, reverse=True)
            m1 = (lab == regs[0].label)
        m1 = binary_fill_holes(m1)
        candidates.append((name, m1))
    act_blur = gaussian(activity_map, sigma=3)
    try: _build("activity_li", act_blur > threshold_li(act_blur),
                opening_disk_px, closing_disk_px)
    except Exception: pass
    for pct in [90, 80, 70, 60]:
        thr = float(np.percentile(act_blur, pct))
        _build(f"activity_p{pct}", act_blur > thr,
               opening_disk_px, closing_disk_px)
    mean_blur = gaussian(mean_frame, sigma=3)
    for name, fn in [("mean_li", threshold_li), ("mean_otsu", threshold_otsu),
                      ("mean_triangle", threshold_triangle)]:
        try: _build(name, mean_blur > fn(mean_blur),
                    opening_disk_px, closing_disk_px)
        except Exception: pass
    for pct in [85, 75, 65, 50]:
        thr = float(np.percentile(mean_blur, pct))
        _build(f"mean_p{pct}", mean_blur > thr,
               opening_disk_px, closing_disk_px)
    if not candidates: return None, "no_candidates"
    scored = []
    for name, m in candidates:
        s = _score_candidate(m, H, W, expected_area_frac,
                             area_frac_range, aspect_range, anchor_yx)
        if s is not None: scored.append((s, name, m))
    if not scored: return None, "all_rejected"
    scored.sort(key=lambda t: -t[0])
    best_score, best_name, best_mask = scored[0]
    if use_convex_hull: best_mask = convex_hull_image(best_mask)
    return best_mask, best_name


def process_and_verify_roi(file_path, fps=None, show_plots=True, save_csv=False,
                            tissue_roi=None,
                            bg_ring_inner_px=15, bg_ring_outer_px=35,
                            min_beat_interval_s=0.4, edge_guard_s=0.5,
                            edge_guard_max_frac=0.15,
                            mid_transient_gap_s=0.4,
                            mid_transient_frac=0.25,
                            smooth_ms=40.0, smooth_polyorder=1,
                            use_convex_hull=True,
                            expected_area_frac=0.30,
                            area_frac_range=(0.05, 0.75),
                            aspect_range=(1.5, 15.0)):
    with nd2.ND2File(file_path) as nd_file:
        video = nd_file.asarray()
        fps_meta = _read_fps_from_nd2(nd_file)
    if fps is None:
        if fps_meta is None:
            raise RuntimeError(f"No fps in {file_path}; pass fps=")
        fps = fps_meta

    v32 = video.astype(np.float32, copy=False)
    mean_frame = v32.mean(axis=0)
    mean_sq = (v32 * v32).mean(axis=0)
    variance = mean_sq - mean_frame * mean_frame
    np.maximum(variance, 0, out=variance)
    activity_map = np.sqrt(variance)

    if tissue_roi is not None:
        y1, y2, x1, x2 = tissue_roi
        H, W = activity_map.shape
        tissue_mask = np.zeros((H, W), dtype=bool)
        tissue_mask[y1:y2, x1:x2] = True
        tissue_source = f"manual_roi"
    else:
        tissue_mask, tissue_source = _detect_tissue_mask(
            v32, activity_map, use_convex_hull=use_convex_hull,
            expected_area_frac=expected_area_frac,
            area_frac_range=area_frac_range, aspect_range=aspect_range)
        if tissue_mask is None:
            raise RuntimeError(f"No tissue in {os.path.basename(file_path)}: "
                               f"{tissue_source}")

    dist_out = distance_transform_edt(~tissue_mask)
    bg_mask = (dist_out > bg_ring_inner_px) & (dist_out <= bg_ring_outer_px)
    bg_mask = bg_mask & (~tissue_mask)
    if bg_mask.sum() < 50:
        bg_mask = (dist_out > bg_ring_inner_px) & (~tissue_mask)
    if bg_mask.sum() < 50:
        raise RuntimeError(f"BG ring too small in "
                           f"{os.path.basename(file_path)}")

    flat32 = v32.reshape(v32.shape[0], -1)
    tf = tissue_mask.ravel().astype(np.float32)
    bf = bg_mask.ravel().astype(np.float32)
    raw_tissue = (flat32 @ tf) / float(tf.sum())
    raw_bg = (flat32 @ bf) / float(bf.sum())
    corrected = raw_tissue - raw_bg
    n = len(corrected)

    window_frames = min(int(fps * 5), max(n // 3, 30))
    dyn_F0 = pd.Series(corrected).rolling(
        window=window_frames, center=True, min_periods=1
    ).quantile(0.10).to_numpy()
    dyn_F0 = pd.Series(dyn_F0).bfill().ffill().to_numpy()
    median_F = float(np.median(raw_tissue))
    F0_floor = max(0.01 * median_F, 1e-6)
    F0_safe = np.maximum(dyn_F0, F0_floor)
    dF_F0_raw = (corrected - F0_safe) / F0_safe

    if smooth_ms and smooth_ms > 0:
        dF_F0, win = _smooth_trace(dF_F0_raw, fps, smooth_ms, smooth_polyorder)
    else:
        dF_F0, win = dF_F0_raw, 0

    noise = _robust_baseline_noise(dF_F0)
    prom = max(0.02, 5.0 * noise)
    eg = int(fps * edge_guard_s)
    if eg > int(n * edge_guard_max_frac): eg = 0
    if n > 2 * eg + 10 and eg > 0:
        lo, hi = eg, n - eg
    else:
        lo, hi = 0, n
    inner = dF_F0[lo:hi]
    inner_peaks, _ = find_peaks(inner, prominence=prom,
                                 distance=int(fps * min_beat_interval_s))
    peaks = inner_peaks + lo
    amplitudes = dF_F0[peaks] if len(peaks) else np.array([])

    if len(peaks) > 0:
        w50, wh50, li50, ri50 = peak_widths(dF_F0, peaks, rel_height=0.5)
        w90, wh90, li90, ri90 = peak_widths(dF_F0, peaks, rel_height=0.9)
        T50_ms = (w50 / fps) * 1000.0
        T90_ms = (w90 / fps) * 1000.0
    else:
        T50_ms = T90_ms = np.array([])

    if len(peaks) >= 2:
        intervals = np.diff(peaks) / fps
        med_int = float(np.median(intervals))
        beat_rate = 1.0 / med_int if med_int > 0 else float("nan")
        rmssd = (float(np.sqrt(np.mean(np.diff(intervals) ** 2)) * 1000)
                 if len(intervals) >= 2 else float("nan"))
        cv = (float(np.std(intervals) / np.mean(intervals))
              if np.mean(intervals) > 0 else float("nan"))
    else:
        intervals = np.array([]); med_int = float("nan")
        beat_rate = float("nan"); rmssd = float("nan"); cv = float("nan")

    tau_ms = np.full(len(peaks), np.nan)
    rise_ms = np.full(len(peaks), np.nan)
    for k, p in enumerate(peaks):
        prv = peaks[k - 1] if k > 0 else None
        nxt = peaks[k + 1] if k + 1 < len(peaks) else None
        tau_ms[k] = _fit_decay_tau(dF_F0, p, fps, next_peak_idx=nxt)
        rise_ms[k] = _rise_time_10_90(dF_F0, p, fps, prev_peak_idx=prv)

    # Drop peak 1 if mid-transient (peak at start or no diastole before it)
    mid_mask = np.zeros(len(peaks), dtype=bool)
    if len(peaks):
        p0 = int(peaks[0]); pv = float(amplitudes[0])
        pre = dF_F0[:p0 + 1]
        pre_min = float(pre.min()) if pre.size else float("inf")
        if (p0 / fps) < mid_transient_gap_s or pre_min > mid_transient_frac * pv:
            mid_mask[0] = True
    keep = ~mid_mask
    def _safe(a, fn):
        a = np.asarray(a)
        if a.size == 0: return float("nan")
        v = fn(a); return float(v) if np.isfinite(v) else float("nan")

    metrics = {
        "path": file_path,
        "duration_s": n / fps, "fps": fps, "fps_metadata": fps_meta,
        "n_peaks": int(len(peaks)),
        "n_peaks_kept": int(keep.sum()),
        "beat_rate_hz": beat_rate,
        "beat_rate_bpm": beat_rate * 60.0 if np.isfinite(beat_rate) else float("nan"),
        "median_interval_s": med_int, "interval_cv": cv, "rmssd_ms": rmssd,
        "amp_mean": _safe(amplitudes[keep] if len(peaks) else [], np.mean),
        "amp_median": _safe(amplitudes[keep] if len(peaks) else [], np.median),
        "T50_ms_mean": _safe(T50_ms[keep] if len(peaks) else [], np.mean),
        "T50_ms_median": _safe(T50_ms[keep] if len(peaks) else [], np.median),
        "T90_ms_mean": _safe(T90_ms[keep] if len(peaks) else [], np.mean),
        "T90_ms_median": _safe(T90_ms[keep] if len(peaks) else [], np.median),
        "tau_ms_median": _safe(tau_ms[keep] if len(peaks) else [], np.nanmedian),
        "rise_ms_median": _safe(rise_ms[keep] if len(peaks) else [], np.nanmedian),
        "diastolic_F": float(np.median(F0_safe)),
        "tissue_area_px": int(tissue_mask.sum()),
        "bg_area_px": int(bg_mask.sum()),
        "noise_MAD": noise,
        "tissue_source": str(tissue_source),
        "upstroke_rate": _upstroke_rate(dF_F0, peaks, fps),
        "iri_median": _iri(dF_F0, peaks, fps),
    }

    if show_plots:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        ax1.imshow(activity_map, cmap="gray")
        ax1.contour(tissue_mask, colors="red", linewidths=1.2)
        ax1.contour(bg_mask, colors="yellow", linewidths=0.5, alpha=0.7)
        ax1.set_title(f"Tissue (red) + BG ring (yellow) — {tissue_source}")
        ax1.axis("off")
        t_axis = np.arange(n) / fps
        ax2.plot(t_axis, dF_F0, color="#1f77b4", label="dF/F0")
        if len(peaks):
            ax2.plot(t_axis[peaks[keep]], amplitudes[keep], "x", color="red",
                     label="Peaks")
        ax2.axhline(prom, color="orange", linestyle=":", alpha=0.6,
                    label=f"prom={prom:.3f}")
        ax2.set_xlabel("s"); ax2.set_ylabel("dF/F0")
        ax2.set_title(f"n={len(peaks)}pk, {beat_rate:.2f} Hz, "
                      f"T50={metrics['T50_ms_median']:.0f}ms, "
                      f"T90={metrics['T90_ms_median']:.0f}ms")
        ax2.legend(fontsize=8)
        plt.tight_layout(); plt.show()

    return {
        "metrics": metrics, "dF_F0": dF_F0, "dF_F0_raw": dF_F0_raw,
        "F0": F0_safe, "raw_tissue": raw_tissue, "raw_bg": raw_bg,
        "peaks": peaks, "amplitudes": amplitudes,
        "T50_ms": T50_ms, "T90_ms": T90_ms,
        "tau_ms": tau_ms, "rise_ms": rise_ms,
        "activity_map": activity_map,
        "tissue_mask": tissue_mask, "bg_mask": bg_mask,
    }


print("process_and_verify_roi() defined.")

# ============================================================
# SHARED batch-progress helpers (used by contact sheet + batch)
# Defined here so any downstream cell can use them regardless of
# whether the contact-sheet cell has been run.
# ============================================================
import threading
_prog_lock = threading.Lock()
_prog = {"done": 0, "total": 0, "t0": None}

def _tick(bn, status):
    with _prog_lock:
        _prog["done"] += 1
        d = _prog["done"]; T = _prog["total"]
        t0 = _prog["t0"] or time.perf_counter()
        dt = time.perf_counter() - t0
        eta = (dt / d) * (T - d) if d else 0.0
        print(f"  [{d:>3}/{T}]  {status:<14}  {dt:>4.0f}s / ~{eta:>4.0f}s   {bn}")


## STEP 1 — Configure the experiment

**Edit the constants below for a new experiment:**

- `BASE` — top-level directory containing your data
- `TIMEPOINT_SUBFOLDERS` — how to map subfolder names to Baseline/Post (per line)
- `PAIR_MAP` — one entry per (line, timepoint, filename-substring, experiment-label); needed only when a folder holds multiple experiments distinguishable by filename. Leave empty to accept every file in each folder.
- `LINE_TO_GROUP` — optional grouping for aggregate stats (e.g. control vs KO)

In [ ]:
# ============================================================
# EDIT THIS BLOCK FOR YOUR EXPERIMENT
# ============================================================
BASE = "/Users/ersm/Desktop/CA Analysis 2"

# Cell lines you're analyzing (folder names under BASE)
LINES = ["MINT", "GEM12"]

# Timepoints and how they map to subfolder names, per line.
# The subfolder name is what actually lives under BASE/<LINE>/.
TIMEPOINT_SUBFOLDERS = {
    "MINT":  {"Baseline": "Baseline", "Post": "Post"},
    "GEM12": {"Baseline": "Baseline", "Post": "Post"},
}

# Optional grouping for aggregate stats
LINE_TO_GROUP = {
    "MINT":  "control",
    "GEM12": "dsp",
}

# PAIR_MAP: only needed when a single subfolder contains multiple
# experiments distinguished by filename substring. Each entry:
#   (line, timepoint, filename_substring, experiment_label)
#
# Files sharing the same experiment_label pair baseline<->post by
# trailing tissue-id digits. Tissue ids become "{experiment}_{digits}".
#
# Leave as [] to accept every file, using tissue-id digits alone as
# the pair key (experiment defaults to "exp").
PAIR_MAP = [
    # Example (from MINT vs GEM12 analysis):
    # ("MINT",  "Baseline", "MINT_calbryte_bl",    "expA"),
    # ("MINT",  "Post",     "MINT_calbryte_30min", "expA"),
    # ("GEM12", "Baseline", "GEM12_bl_calbryte",   "expB"),
    # ("GEM12", "Post",     "GEM12_postET1",       "expB"),
]

# Analysis knobs (usually leave as-is)
ANALYSIS_KWARGS = dict(
    smooth_ms=40.0,
    smooth_polyorder=1,
    expected_area_frac=0.30,
    area_frac_range=(0.05, 0.75),
    aspect_range=(1.5, 15.0),
    use_convex_hull=True,
)

# Persistent state files
MANUAL_ROIS_PATH = os.path.expanduser("~/calcium_manual_rois.json")
EXCLUDE_PATH     = os.path.expanduser("~/calcium_qc_exclude.json")
CACHE_DIR        = os.path.expanduser("~/calcium_runs_cache")
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"BASE            = {BASE}")
print(f"LINES           = {LINES}")
print(f"PAIR_MAP        = {len(PAIR_MAP)} entries")
print(f"Manual ROIs at  = {MANUAL_ROIS_PATH}")
print(f"Exclusions at   = {EXCLUDE_PATH}")
print(f"Cache dir       = {CACHE_DIR}")

## STEP 2 — Discover files and pair by tissue ID

In [ ]:
import re, glob, json
from collections import defaultdict

TIMEPOINTS = ["Baseline", "Post"]
_ID_RE = re.compile(r"(?:^|[_ ])(\d{2,4})(?:\+[Cc][Aa]\d*)?\.nd2$")

def _extract_tid(basename):
    m = _ID_RE.search(basename)
    return m.group(1) if m else None

def _match_experiment(line, timepoint, basename):
    if not PAIR_MAP:
        return "exp"       # default single-experiment mode
    bn_low = basename.lower()
    matches = []
    for L, tp, patt, exp in PAIR_MAP:
        if L != line or tp != timepoint: continue
        if patt.lower() in bn_low: matches.append((len(patt), exp))
    if not matches: return None
    matches.sort(reverse=True)
    return matches[0][1]

index = {}
unmatched = []
for line in LINES:
    for tp in TIMEPOINTS:
        sub = TIMEPOINT_SUBFOLDERS.get(line, {}).get(tp, tp)
        d = os.path.join(BASE, line, sub)
        by_tid = {}
        if not os.path.isdir(d):
            print(f"WARNING: missing directory {d}")
        else:
            for p in sorted(glob.glob(os.path.join(d, "*.nd2"))):
                bn = os.path.basename(p)
                digits = _extract_tid(bn)
                if digits is None:
                    print(f"WARNING: could not parse tissue id from {bn}; skipping")
                    continue
                exp = _match_experiment(line, tp, bn)
                if exp is None:
                    unmatched.append((line, tp, bn)); continue
                tid = f"{exp}_{digits}"
                if tid in by_tid:
                    print(f"DUP {line}/{tp} {tid}: keeping "
                          f"{os.path.basename(by_tid[tid])}, skipping {bn}")
                    continue
                by_tid[tid] = p
        index[(line, tp)] = by_tid

if unmatched:
    print(f"\n{len(unmatched)} unmatched files — add a PAIR_MAP entry:")
    for line, tp, bn in unmatched[:20]:
        print(f"  {line}/{tp}  {bn}")

# Pair BL <-> Post by tissue id
print("\n" + "=" * 78)
print(f"{'line':<8}{'baseline':>10}{'post':>10}{'paired':>10}{'BL-only':>10}{'post-only':>12}")
print("-" * 78)
paired_ids = {}
for line in LINES:
    bl = set(index.get((line, "Baseline"), {}).keys())
    po = set(index.get((line, "Post"), {}).keys())
    both = sorted(bl & po)
    paired_ids[line] = both
    print(f"{line:<8}{len(bl):>10}{len(po):>10}{len(both):>10}"
          f"{len(bl-po):>10}{len(po-bl):>12}")
print("=" * 78)

# Load persistent manual ROIs
if os.path.exists(MANUAL_ROIS_PATH):
    with open(MANUAL_ROIS_PATH) as f:
        MANUAL_ROIS = {k: tuple(v) for k, v in json.load(f).items()}
    print(f"\nLoaded {len(MANUAL_ROIS)} manual ROI(s) from {MANUAL_ROIS_PATH}")
else:
    MANUAL_ROIS = {}
    print(f"\nNo manual ROIs at {MANUAL_ROIS_PATH} yet.")

def save_manual_rois():
    with open(MANUAL_ROIS_PATH, "w") as f:
        json.dump({k: list(v) for k, v in MANUAL_ROIS.items()}, f, indent=2)
    print(f"Saved {len(MANUAL_ROIS)} manual ROI(s).")

# List of all (line, tp, tid, path) tuples
ALL_FILES = []
for line in LINES:
    for tp in TIMEPOINTS:
        for tid, p in sorted(index.get((line, tp), {}).items()):
            ALL_FILES.append((line, tp, tid, p))
print(f"\nTotal files to analyze: {len(ALL_FILES)}")

## STEP 3 — Contact sheet: check auto-ROI quality

Renders a thumbnail of every file with the auto-detected tissue outline in red. Note which tissue IDs have bad ROIs — you'll fix them in STEP 4.

In [ ]:
import time, math, threading, traceback
from concurrent.futures import ThreadPoolExecutor, as_completed

CONTACT_MAX_WORKERS = 4
N_SAMPLE_FRAMES     = 12
THUMB_MAX_W         = 500
THUMB_COLS          = 6

def _read_sampled(path, n_frames=N_SAMPLE_FRAMES):
    """Return (mean_img, activity_img) from n_frames evenly-spaced frames."""
    with nd2.ND2File(path) as nf:
        d = nf.to_dask()
        if d.ndim == 4: d = d[:, 0]
        T = d.shape[0]
        idxs = (list(range(T)) if T <= n_frames
                else np.linspace(0, T - 1, n_frames).astype(int).tolist())
        mean_img = None; M2 = None; count = 0
        for i in idxs:
            f = np.asarray(d[i], dtype=np.float32); count += 1
            if mean_img is None:
                mean_img = f.copy(); M2 = np.zeros_like(f)
            else:
                delta = f - mean_img
                mean_img += delta / count
                M2 += delta * (f - mean_img)
        variance = M2 / max(count, 1)
        np.maximum(variance, 0, out=variance)
        activity = np.sqrt(variance)
    return mean_img, activity

def _contact_work(item):
    line, tp, tid, path = item
    bn = os.path.basename(path)
    try:
        mean, activity = _read_sampled(path)
    except Exception as e:
        _tick(bn, f"LOAD-ERR:{type(e).__name__}")
        return line, tp, tid, path, None, None, False, "load-error"
    if bn in MANUAL_ROIS:
        H, W = activity.shape
        y1, y2, x1, x2 = MANUAL_ROIS[bn]
        m = np.zeros((H, W), dtype=bool); m[y1:y2, x1:x2] = True
        _tick(bn, "manual")
        return line, tp, tid, path, mean, m, True, "manual"
    try:
        fake_stack = np.stack([mean, mean], axis=0)
        m, src = _detect_tissue_mask(fake_stack, activity,
                                      **{k: v for k, v in ANALYSIS_KWARGS.items()
                                         if k in ("expected_area_frac",
                                                  "area_frac_range",
                                                  "aspect_range",
                                                  "use_convex_hull")})
    except Exception as e:
        _tick(bn, f"DET-ERR")
        return line, tp, tid, path, mean, None, False, "detect-error"
    if m is None:
        _tick(bn, "no-tissue")
        return line, tp, tid, path, mean, None, False, str(src)
    frac = float(m.sum()) / m.size
    ys, xs = np.where(m)
    h = ys.max() - ys.min() + 1; w = xs.max() - xs.min() + 1
    aspect = max(h, w) / max(1, min(h, w))
    ok = True; warn = ""
    if frac < 0.05 or frac > 0.75:
        ok = False; warn = f"area={frac:.0%}"
    elif aspect < 1.5:
        ok = False; warn = f"aspect={aspect:.1f}"
    _tick(bn, "OK" if ok else warn)
    return line, tp, tid, path, mean, m, ok, warn

_prog.update(done=0, total=len(ALL_FILES), t0=time.perf_counter())
print(f"Contact sheet: {len(ALL_FILES)} files, {CONTACT_MAX_WORKERS} workers")
contact_results = []
with ThreadPoolExecutor(max_workers=CONTACT_MAX_WORKERS) as ex:
    for fut in as_completed(ex.submit(_contact_work, it) for it in ALL_FILES):
        contact_results.append(fut.result())

contact_results.sort(key=lambda r: (r[0], r[1], r[2]))
sus = [(line, tp, tid, path, warn)
       for line, tp, tid, path, mean, m, ok, warn in contact_results if not ok]
print(f"\nAUTO-ROI QC:  OK = {len(contact_results) - len(sus)}   "
      f"suspicious = {len(sus)}")
if sus:
    print("Files flagged for manual ROI:")
    for l, tp, tid, path, w in sus:
        print(f"  {l:<6} {tp:<9} {tid:<10}  {w:<12}  {os.path.basename(path)}")

# Thumbnail grid per (line, tp)
groups = defaultdict(list)
for r in contact_results: groups[(r[0], r[1])].append(r)
for (line, tp), items in sorted(groups.items()):
    n = len(items); cols = min(THUMB_COLS, n); rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.4))
    axes = np.atleast_2d(axes).ravel()
    for i, (_, _, tid, path, mean, m, ok, warn) in enumerate(items):
        ax = axes[i]
        if mean is not None:
            H, W = mean.shape
            step = max(1, W // THUMB_MAX_W)
            ax.imshow(mean[::step, ::step], cmap="gray",
                      vmin=float(np.percentile(mean, 2)),
                      vmax=float(np.percentile(mean, 99.5)))
            if m is not None:
                ax.contour(m[::step, ::step], colors="red", linewidths=0.9)
        ax.set_title(f"{tid}\n{'OK' if ok else warn}", fontsize=8,
                     color="black" if ok else "red")
        ax.axis("off")
    for j in range(len(items), len(axes)): axes[j].axis("off")
    fig.suptitle(f"{line} / {tp}   (n={n})", fontweight="bold")
    plt.tight_layout(); plt.show()

SUS_LIST = [(l, tp, tid, path) for (l, tp, tid, path, w) in sus]
print(f"\n{len(SUS_LIST)} file(s) need manual ROI review (see STEP 4).")

## STEP 4 — Manual ROI picker (optional)

Skip if all ROIs looked good in STEP 3. Otherwise, run the cell below and drag a rectangle around the tissue in each flagged file. ROIs are saved to `~/calcium_manual_rois.json` and applied automatically in STEP 5.

**Add any additional bad-ROI files to `EXTRA_MANUAL_LIST` if the auto-QC missed some.**

In [ ]:
# ipywidgets-based ROI picker: click and drag on the mean image.
# Runs one file at a time. Rerun this cell if the widget doesn't respond.
import ipywidgets as widgets
from IPython.display import display

EXTRA_MANUAL_LIST = []           # add (line, tp, tid, path) tuples here if needed
TO_PICK = SUS_LIST + EXTRA_MANUAL_LIST

if not TO_PICK:
    print("No files need manual ROIs.")
else:
    print(f"Will pick ROIs for {len(TO_PICK)} files.")
    print("For each: run `pick_next()` in a fresh cell, drag a rectangle,\n"
          "then click 'Save & next'. Loops until list is empty.")

    _iter_state = {"i": 0}

    def _pick_one(line, tp, tid, path):
        bn = os.path.basename(path)
        mean, _ = _read_sampled(path)
        H, W = mean.shape
        fig, ax = plt.subplots(figsize=(10, 4.5))
        ax.imshow(mean, cmap="gray",
                  vmin=float(np.percentile(mean, 2)),
                  vmax=float(np.percentile(mean, 99.5)))
        ax.set_title(f"{line} / {tp} — {tid}\n{bn}"); ax.axis("off")
        y1 = widgets.IntText(value=0,   description="y1:", layout=widgets.Layout(width="150px"))
        y2 = widgets.IntText(value=H-1, description="y2:", layout=widgets.Layout(width="150px"))
        x1 = widgets.IntText(value=0,   description="x1:", layout=widgets.Layout(width="150px"))
        x2 = widgets.IntText(value=W-1, description="x2:", layout=widgets.Layout(width="150px"))
        preview = widgets.Output()
        def _update_preview(*_):
            with preview:
                preview.clear_output(wait=True)
                fig2, ax2 = plt.subplots(figsize=(10, 4.5))
                ax2.imshow(mean, cmap="gray",
                            vmin=float(np.percentile(mean, 2)),
                            vmax=float(np.percentile(mean, 99.5)))
                y1v, y2v, x1v, x2v = y1.value, y2.value, x1.value, x2.value
                rect = plt.Rectangle((x1v, y1v), x2v - x1v, y2v - y1v,
                                     fill=False, edgecolor="red", linewidth=2)
                ax2.add_patch(rect); ax2.axis("off")
                plt.show()
        for w in (y1, y2, x1, x2): w.observe(_update_preview, names="value")
        save_btn = widgets.Button(description="Save & next", button_style="primary")
        skip_btn = widgets.Button(description="Skip")
        status = widgets.HTML()
        def _save(_):
            MANUAL_ROIS[bn] = (y1.value, y2.value, x1.value, x2.value)
            save_manual_rois()
            status.value = f"<b>Saved {bn}</b>. Run pick_next()."
        def _skip(_):
            status.value = f"Skipped {bn}. Run pick_next()."
        save_btn.on_click(_save); skip_btn.on_click(_skip)
        display(widgets.VBox([widgets.HBox([y1, y2, x1, x2]),
                              widgets.HBox([save_btn, skip_btn]),
                              status, preview]))
        _update_preview()

    def pick_next():
        i = _iter_state["i"]
        if i >= len(TO_PICK):
            print("All files done. Continue to STEP 5.")
            return
        line, tp, tid, path = TO_PICK[i]
        _iter_state["i"] = i + 1
        print(f"[{i+1}/{len(TO_PICK)}]  {line} {tp} {tid}")
        _pick_one(line, tp, tid, path)

    print("Type `pick_next()` in a new cell below to start.")

## STEP 5 — Batch analysis

Runs the full analysis on every file, parallelized. Uses `~/calcium_runs_cache/` — re-running skips previously-analyzed files. Applies manual ROIs automatically.

In [ ]:
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_MAX_WORKERS = 6

def _cache_key(path, roi):
    st = os.stat(path)
    roi_tag = "auto" if roi is None else "roi_" + "_".join(str(v) for v in roi)
    return f"{os.path.basename(path)}__{int(st.st_mtime)}__{roi_tag}.pkl"

def _load_cached(path, roi):
    fp = os.path.join(CACHE_DIR, _cache_key(path, roi))
    if os.path.exists(fp):
        try:
            with open(fp, "rb") as f: return pickle.load(f)
        except Exception: return None
    return None

def _save_cached(path, roi, result):
    fp = os.path.join(CACHE_DIR, _cache_key(path, roi))
    try:
        with open(fp, "wb") as f: pickle.dump(result, f,
                                                protocol=pickle.HIGHEST_PROTOCOL)
    except Exception as e:
        print(f"  cache save failed for {os.path.basename(path)}: {e}")

_prog.update(done=0, total=len(ALL_FILES), t0=time.perf_counter())

def _analyze(item):
    line, tp, tid, path = item
    bn = os.path.basename(path)
    roi = MANUAL_ROIS.get(bn)
    cached = _load_cached(path, roi)
    if cached is not None:
        _tick(bn, "CACHED"); return (line, tp, tid), cached, None
    t0 = time.perf_counter()
    try:
        result = process_and_verify_roi(path, show_plots=False,
                                          tissue_roi=roi, **ANALYSIS_KWARGS)
    except Exception as e:
        _tick(bn, "FAIL")
        return (line, tp, tid), None, f"{type(e).__name__}: {e}"
    _save_cached(path, roi, result)
    _tick(bn, f"OK({time.perf_counter()-t0:.0f}s)")
    return (line, tp, tid), result, None

print(f"Batch: {len(ALL_FILES)} files, {BATCH_MAX_WORKERS} workers, "
      f"cache at {CACHE_DIR}")
runs = {}
failures = []
with ThreadPoolExecutor(max_workers=BATCH_MAX_WORKERS) as ex:
    for fut in as_completed(ex.submit(_analyze, it) for it in ALL_FILES):
        key, result, err = fut.result()
        if err: failures.append((key, err)); continue
        runs[key] = result

print(f"\nBatch complete: {len(runs)} runs   {len(failures)} failures")
if failures:
    for key, err in failures[:10]:
        print(f"  FAIL {key}: {err}")

## STEP 6 — Trace QC picker

Click "Exclude" on any bad trace; hit Save to write `~/calcium_qc_exclude.json`. Exclusions are applied in STEP 7.

In [ ]:
%matplotlib inline
import io, matplotlib
matplotlib.use("Agg", force=True)
import ipywidgets as widgets
from IPython.display import display

if os.path.exists(EXCLUDE_PATH):
    with open(EXCLUDE_PATH) as f: EXCLUDE_LIST = set(json.load(f))
else:
    EXCLUDE_LIST = set()

RUN_KEYS = sorted(runs.keys(), key=lambda k: (k[0], k[1], k[2]))
already = sum(1 for k in RUN_KEYS
              if os.path.basename(runs[k]["metrics"]["path"]) in EXCLUDE_LIST)
print(f"Showing {len(RUN_KEYS)} traces ({already} already excluded).")

def _trace_png(dF, fps, peaks, w=2.6, h=1.2):
    fig, ax = plt.subplots(figsize=(w, h), dpi=90)
    t = np.arange(len(dF)) / fps
    ax.plot(t, dF, color="#1f77b4", linewidth=0.7)
    if len(peaks): ax.plot(peaks / fps, dF[peaks], "x", color="red", markersize=3)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    ax.margins(x=0.01, y=0.05)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)
    return buf.getvalue()

cards, cb_map = [], {}
for key in RUN_KEYS:
    line, tp, tid = key
    r = runs[key]
    bn = os.path.basename(r["metrics"]["path"])
    try: png = _trace_png(r["dF_F0"], r["metrics"]["fps"], r["peaks"])
    except Exception as e:
        print(f"  render failed {bn}: {e}"); png = b""
    img = widgets.Image(value=png, format="png",
                        layout=widgets.Layout(width="280px"))
    cb = widgets.Checkbox(value=(bn in EXCLUDE_LIST),
                          description=f"Exclude {tid}", indent=False)
    cb_map[bn] = cb
    m = r["metrics"]
    label = widgets.HTML(
        f"<div style='font-size:11px'><b>{line} / {tp}</b> — {tid} — "
        f"{m['n_peaks']}pk, {m['beat_rate_hz']:.2f} Hz<br>"
        f"<span style='color:#888'>{bn}</span></div>")
    cards.append(widgets.VBox([img, label, cb],
                               layout=widgets.Layout(border="1px solid #ddd",
                                                     padding="4px", margin="2px",
                                                     width="300px")))

grid = widgets.GridBox(cards, layout=widgets.Layout(
    grid_template_columns="repeat(4, 310px)"))
count = widgets.HTML(); status = widgets.HTML()
save_btn = widgets.Button(description="Save exclusions", button_style="primary")

def _refresh(*_):
    n_ex = sum(1 for cb in cb_map.values() if cb.value)
    n_kept = len({b for b in EXCLUDE_LIST if b not in cb_map})
    count.value = (f"<b>{n_ex}</b> selected for exclusion "
                   f"({n_kept} other-experiment exclusions preserved).")
for cb in cb_map.values(): cb.observe(_refresh, names="value")
_refresh()

def _save(_):
    non = {b for b in EXCLUDE_LIST if b not in cb_map}
    excl = {bn for bn, cb in cb_map.items() if cb.value}
    new = sorted(non | excl)
    with open(EXCLUDE_PATH, "w") as f: json.dump(new, f, indent=2)
    globals()["EXCLUDE_LIST"] = set(new)
    status.value = (f"<span style='color:green'>Saved {len(new)} exclusions "
                    f"to {EXCLUDE_PATH}</span>")

save_btn.on_click(_save)
display(widgets.VBox([widgets.HTML("<h4>Click bad traces, hit Save.</h4>"),
                       count, save_btn, status, grid]))

## STEP 7 — Stats + composite figure

Builds paired arrays across all lines / experiments (respecting the QC exclusion list), prints paired within-line + between-line stats, and saves the composite figure as SVG (editable in Affinity) + TIFF.

In [ ]:
%matplotlib inline
import math
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

# Output paths — timestamp so re-running doesn't overwrite prior versions
_stamp = time.strftime("%Y%m%d_%H%M%S")
FIG_SVG   = os.path.expanduser(f"~/calcium_composite_{_stamp}.svg")
FIG_TIFF  = os.path.expanduser(f"~/calcium_composite_{_stamp}.tiff")
FIG_DPI = 300

# Reload exclusions
if os.path.exists(EXCLUDE_PATH):
    with open(EXCLUDE_PATH) as f: EXCLUDE_LIST = set(json.load(f))
else:
    EXCLUDE_LIST = set()

# Colors: assigned to lines in the order they appear in LINES
_PALETTE = ["#4477aa", "#ee6677", "#228833", "#ccbb44", "#66ccee", "#aa3377"]
COLORS = {line: _PALETTE[i % len(_PALETTE)] for i, line in enumerate(LINES)}

METRIC_KEYS = ["beat_rate_hz", "upstroke_rate",
               "T50_ms_median", "T90_ms_median", "iri_median"]

def _keep(k):
    return (k in runs
            and os.path.basename(runs[k]["metrics"]["path"]) not in EXCLUDE_LIST)

paired = {}
for line in LINES:
    tids = [t for t in paired_ids.get(line, [])
            if _keep((line, "Baseline", t)) and _keep((line, "Post", t))]
    for k in METRIC_KEYS:
        bl = np.array([runs[(line, "Baseline", t)]["metrics"].get(k, float("nan"))
                       for t in tids], dtype=float)
        po = np.array([runs[(line, "Post",     t)]["metrics"].get(k, float("nan"))
                       for t in tids], dtype=float)
        paired[(line, k)] = (bl, po, tids)

# Console stats
def _fp(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); return a[m], b[m]

def _wil(a, b):
    a, b = _fp(a, b)
    if a.size < 2 or np.allclose(a, b): return float("nan")
    try: return float(sps.wilcoxon(a, b).pvalue)
    except: return float("nan")

def _mwu(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size < 2 or b.size < 2: return float("nan")
    try: return float(sps.mannwhitneyu(a, b, alternative="two-sided").pvalue)
    except: return float("nan")

def _dz(a, b):
    a, b = _fp(a, b); d = b - a
    if d.size < 2 or d.std(ddof=1) == 0: return float("nan")
    return float(d.mean() / d.std(ddof=1))

def _cliffs(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a[np.isfinite(a)]; b = b[np.isfinite(b)]
    if a.size == 0 or b.size == 0: return float("nan")
    gt = (a[:, None] > b[None, :]).sum()
    lt = (a[:, None] < b[None, :]).sum()
    return float((gt - lt) / (a.size * b.size))

def _fmt_p(p):
    if not np.isfinite(p): return "n.s."
    if p < 1e-4: return "p<1e-4"
    if p < 1e-3: return f"p={p:.1e}"
    if p < 0.05: return f"p={p:.3f}"
    return f"p={p:.2f}"

print(f"Post-QC sample sizes ({len(EXCLUDE_LIST)} exclusions applied):")
for line in LINES:
    n = int(np.isfinite(paired[(line, 'beat_rate_hz')][0]).sum())
    print(f"  {line:<8}  n = {n}")

print("\n" + "=" * 78)
print("METRIC SUMMARY")
print("=" * 78)
for k, lbl in [("beat_rate_hz",   "Beat rate (Hz)"),
                ("upstroke_rate",  "Upstroke rate (dF/F0/s)"),
                ("T50_ms_median",  "CaTD50 (ms)"),
                ("T90_ms_median",  "CaTD90 (ms)"),
                ("iri_median",     "IRI")]:
    print(f"\n  {lbl}")
    for line in LINES:
        bl, po, _ = paired[(line, k)]
        bl_f, po_f = _fp(bl, po)
        med_bl = float(np.nanmedian(bl_f)) if bl_f.size else float("nan")
        med_po = float(np.nanmedian(po_f)) if po_f.size else float("nan")
        p = _wil(bl_f, po_f); dz = _dz(bl_f, po_f)
        print(f"    {line:<8}  n={bl_f.size:2d}   BL={med_bl:7.2f}  Post={med_po:7.2f}  "
              f"Δ={med_po-med_bl:+7.2f}   Wilcoxon {_fmt_p(p)}   dz={dz:+.2f}")
    if len(LINES) == 2:
        L1, L2 = LINES
        dm = paired[(L1, k)][1] - paired[(L1, k)][0]
        dg = paired[(L2, k)][1] - paired[(L2, k)][0]
        print(f"    Δ {L1} vs {L2}:  MWU {_fmt_p(_mwu(dm, dg))}   "
              f"δ={_cliffs(dm, dg):+.2f}")

# --- Figure (5-panel composite, only if exactly 2 lines) ---
if len(LINES) == 2:
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
        "font.size": 10, "svg.fonttype": "none", "pdf.fonttype": 42,
    })
    L1, L2 = LINES

    def _draw_paired(ax, k, ylabel, title, letter):
        x = np.array([0, 1]); ymax = []
        for li, line in enumerate(LINES):
            bl, po, _ = paired[(line, k)]
            blf, pof = _fp(bl, po)
            if blf.size == 0: continue
            xoff = x + (li - 0.5) * 0.10
            for a, b in zip(blf, pof):
                ax.plot(xoff, [a, b], "o-", color=COLORS[line],
                        alpha=0.3, linewidth=0.9, markersize=3.5,
                        markeredgecolor="none")
            for i, arr in enumerate([blf, pof]):
                mu = float(arr.mean())
                sem = float(arr.std(ddof=1) / math.sqrt(arr.size)) if arr.size > 1 else 0
                ax.errorbar([xoff[i]], [mu], yerr=[sem], fmt="s",
                            color=COLORS[line], markersize=8,
                            markeredgecolor="black", markeredgewidth=0.8,
                            capsize=4, capthick=1.2, elinewidth=1.4, zorder=10)
            ymax.append(max(blf.max(), pof.max()))
        for li, line in enumerate(LINES):
            bl, po, _ = paired[(line, k)]
            blf, pof = _fp(bl, po)
            if blf.size == 0: continue
            p = _wil(blf, pof)
            y_top = max(ymax); y_pad = y_top * 0.03
            y_bar = y_top + y_pad * (2 + li * 1.8)
            xoff = x + (li - 0.5) * 0.10
            ax.plot([xoff[0], xoff[0], xoff[1], xoff[1]],
                    [y_bar, y_bar + y_pad, y_bar + y_pad, y_bar],
                    color=COLORS[line], linewidth=1.0)
            ax.text(xoff.mean(), y_bar + y_pad * 1.2, _fmt_p(p),
                    ha="center", va="bottom", fontsize=8, color=COLORS[line])
        ax.set_xticks(x); ax.set_xticklabels(["Baseline", "Post"])
        ax.set_ylabel(ylabel)
        ax.set_title(title, fontweight="bold", loc="left")
        ax.text(-0.13, 1.05, letter, transform=ax.transAxes,
                fontsize=13, fontweight="bold", va="bottom")
        ax.grid(True, alpha=0.3, axis="y")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)

    def _draw_delta(ax, k, ylabel, title, letter, rng):
        ds = {}
        for line in LINES:
            bl, po, _ = paired[(line, k)]
            d = po - bl; ds[line] = d[np.isfinite(d)]
        for li, line in enumerate(LINES):
            d = ds[line]
            if d.size == 0: continue
            j = rng.uniform(-0.14, 0.14, size=d.size)
            ax.plot(np.full_like(d, li, float) + j, d, "o",
                    color=COLORS[line], alpha=0.7, markersize=6,
                    markeredgecolor="black", markeredgewidth=0.4)
            ax.boxplot([d], positions=[li], widths=0.55, showfliers=False,
                       boxprops=dict(color="black", linewidth=1.2),
                       medianprops=dict(color="black", linewidth=2),
                       whiskerprops=dict(color="black"),
                       capprops=dict(color="black"))
        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
        ax.set_xticks(range(len(LINES)))
        ax.set_xticklabels([f"{l}\n(n={ds[l].size})" for l in LINES], fontsize=9)
        ax.set_ylabel(ylabel); ax.set_title(title, fontweight="bold", loc="left")
        ax.text(-0.18, 1.05, letter, transform=ax.transAxes,
                fontsize=13, fontweight="bold", va="bottom")
        ax.grid(True, alpha=0.3, axis="y")
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        if ds[L1].size >= 2 and ds[L2].size >= 2:
            p = _mwu(ds[L1], ds[L2])
            y_top = max(ds[L1].max(), ds[L2].max())
            y_bot = min(ds[L1].min(), ds[L2].min())
            y_pad = (y_top - y_bot) * 0.06
            y_bar = y_top + y_pad
            ax.plot([0, 0, 1, 1],
                    [y_bar, y_bar + y_pad, y_bar + y_pad, y_bar],
                    color="black", linewidth=1.0)
            ax.text(0.5, y_bar + y_pad * 1.3, _fmt_p(p),
                    ha="center", va="bottom", fontsize=9)

    fig = plt.figure(figsize=(12.0, 7.4))
    gs = GridSpec(2, 6, figure=fig, hspace=0.55, wspace=0.9,
                   left=0.06, right=0.98, top=0.92, bottom=0.10)
    axA = fig.add_subplot(gs[0, 0:3]); axB = fig.add_subplot(gs[0, 3:6])
    axC = fig.add_subplot(gs[1, 0:2]); axD = fig.add_subplot(gs[1, 2:4])
    axE = fig.add_subplot(gs[1, 4:6])
    _draw_paired(axA, "beat_rate_hz", "Beat rate (Hz)", "Beat rate", "A")
    _draw_paired(axB, "upstroke_rate", "Upstroke rate", "Upstroke rate", "B")
    rng = np.random.default_rng(0)
    _draw_delta(axC, "T50_ms_median", "Δ CaTD50 (ms)", "Δ CaTD50", "C", rng)
    _draw_delta(axD, "T90_ms_median", "Δ CaTD90 (ms)", "Δ CaTD90", "D", rng)
    _draw_delta(axE, "upstroke_rate", "Δ Upstroke rate", "Δ Upstroke rate", "E", rng)

    handles = [Line2D([0], [0], marker="s", linestyle="-", color=COLORS[l],
                        markeredgecolor="black", markersize=8, markeredgewidth=0.8,
                        label=f"{l} (n={int(np.isfinite(paired[(l,'beat_rate_hz')][0]).sum())})")
               for l in LINES]
    fig.legend(handles=handles, loc="lower center", ncol=len(LINES),
                bbox_to_anchor=(0.5, 0.00), fontsize=10, frameon=False)
    fig.suptitle(f"{L1} vs {L2}  —  calcium response to treatment",
                  fontsize=14, fontweight="bold", y=0.98)
    plt.savefig(FIG_SVG, format="svg", bbox_inches="tight")
    plt.savefig(FIG_TIFF, format="tiff", dpi=FIG_DPI,
                 bbox_inches="tight", pil_kwargs={"compression": "tiff_lzw"})
    plt.show()
    print(f"\n  Figure SVG  -> {FIG_SVG}")
    print(f"  Figure TIFF -> {FIG_TIFF}")
else:
    print(f"\n(Composite figure builder assumes 2 lines — you have {len(LINES)}. "
          f"Console stats above still work.)")

## STEP 8 — Snapshot for archival

Save the full dataset (traces + metrics + stats + notebook backup) to `~/calcium_snapshots/<date>_<name>/`. Edit `SNAPSHOT_NAME` before running.

In [ ]:
import shutil, datetime

SNAPSHOT_NAME = "my_experiment"       # <-- edit for each experiment
SNAPSHOT_DIR = os.path.expanduser(
    f"~/calcium_snapshots/{datetime.date.today().isoformat()}_{SNAPSHOT_NAME}")
os.makedirs(SNAPSHOT_DIR, exist_ok=True)
print(f"Snapshot directory: {SNAPSHOT_DIR}")

# 1. runs.pkl
with open(os.path.join(SNAPSHOT_DIR, "runs.pkl"), "wb") as f:
    pickle.dump(runs, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"  ✓ runs.pkl                      ({len(runs)} entries)")

# 2. per_tissue.csv
rows = []
for (line, tp, tid), r in runs.items():
    m = r["metrics"]
    row = {
        "line": line, "timepoint": tp, "tissue_id": tid,
        "experiment": tid.rsplit("_", 1)[0],
        "basename": os.path.basename(m["path"]),
        "path": m["path"],
        "excluded": os.path.basename(m["path"]) in EXCLUDE_LIST,
    }
    for k, v in m.items():
        if k != "path": row[k] = v
    rows.append(row)
df_t = pd.DataFrame(rows).sort_values(
    ["line", "experiment", "timepoint", "tissue_id"]).reset_index(drop=True)
df_t.to_csv(os.path.join(SNAPSHOT_DIR, "per_tissue.csv"), index=False)
print(f"  ✓ per_tissue.csv                 ({len(df_t)} rows)")

# 3. paired_wide.csv (long-form-ish; one row per pair with BL/Post/Δ)
PAIRED_METRICS = ["beat_rate_hz", "beat_rate_bpm", "amp_median", "amp_mean",
                   "T50_ms_median", "T90_ms_median",
                   "tau_ms_median", "rise_ms_median",
                   "interval_cv", "rmssd_ms",
                   "upstroke_rate", "iri_median"]
pr = []
for line in LINES:
    for tid in paired_ids.get(line, []):
        bl = (line, "Baseline", tid); po = (line, "Post", tid)
        if bl not in runs or po not in runs: continue
        blm, pom = runs[bl]["metrics"], runs[po]["metrics"]
        row = {
            "line": line, "group": LINE_TO_GROUP.get(line, ""),
            "tissue_id": tid, "experiment": tid.rsplit("_", 1)[0],
            "excluded_bl": os.path.basename(blm["path"]) in EXCLUDE_LIST,
            "excluded_po": os.path.basename(pom["path"]) in EXCLUDE_LIST,
            "any_excluded": (os.path.basename(blm["path"]) in EXCLUDE_LIST or
                             os.path.basename(pom["path"]) in EXCLUDE_LIST),
        }
        for k in PAIRED_METRICS:
            bv = blm.get(k, float("nan")); pv = pom.get(k, float("nan"))
            row[f"{k}__BL"] = bv; row[f"{k}__Post"] = pv
            try: row[f"{k}__delta"] = pv - bv
            except Exception: row[f"{k}__delta"] = float("nan")
        pr.append(row)
df_p = pd.DataFrame(pr).sort_values(
    ["line", "experiment", "tissue_id"]).reset_index(drop=True)
df_p.to_csv(os.path.join(SNAPSHOT_DIR, "paired_wide.csv"), index=False)
print(f"  ✓ paired_wide.csv                ({len(df_p)} pairs)")

# 4. exclusions.json + manual_rois.json
with open(os.path.join(SNAPSHOT_DIR, "exclusions.json"), "w") as f:
    json.dump(sorted(EXCLUDE_LIST), f, indent=2)
print(f"  ✓ exclusions.json                ({len(EXCLUDE_LIST)} basenames)")
if os.path.exists(MANUAL_ROIS_PATH):
    shutil.copy(MANUAL_ROIS_PATH,
                os.path.join(SNAPSHOT_DIR, "manual_rois.json"))
    print(f"  ✓ manual_rois.json")

# 5. Copy the current figure files (from STEP 7)
try:
    if os.path.exists(FIG_SVG):
        shutil.copy(FIG_SVG, os.path.join(SNAPSHOT_DIR, os.path.basename(FIG_SVG)))
        print(f"  ✓ {os.path.basename(FIG_SVG)}")
    if os.path.exists(FIG_TIFF):
        shutil.copy(FIG_TIFF, os.path.join(SNAPSHOT_DIR, os.path.basename(FIG_TIFF)))
        print(f"  ✓ {os.path.basename(FIG_TIFF)}")
except NameError:
    pass

# 6. Notebook backup
NB_SRC = "/Users/ersm/Calcium Pipeline.ipynb"
if os.path.exists(NB_SRC):
    shutil.copy(NB_SRC, os.path.join(SNAPSHOT_DIR,
                                       "Calcium_Pipeline_backup.ipynb"))
    print(f"  ✓ Calcium_Pipeline_backup.ipynb")

# 7. README
readme = f"""# Calcium imaging snapshot — {datetime.date.today().isoformat()}

Experiment: {SNAPSHOT_NAME}
Base data: {BASE}
Lines: {LINES}

## Contents
- runs.pkl              — full analysis dict (load with pickle)
- per_tissue.csv        — one row per tissue, all metrics
- paired_wide.csv       — one row per pair, BL/Post/Δ columns
- exclusions.json       — QC-excluded basenames
- manual_rois.json      — manual ROI dict
- Calcium_Pipeline_backup.ipynb — notebook backup
- calcium_composite_*.svg/.tiff — main figure

## Sample sizes (post-QC)
""" + "\n".join(
    f"- {line}:  n = {len(df_p[(df_p['line']==line) & (~df_p['any_excluded'])])}"
    for line in LINES)
with open(os.path.join(SNAPSHOT_DIR, "README.md"), "w") as f:
    f.write(readme)
print(f"  ✓ README.md")

print(f"\nSnapshot complete: {SNAPSHOT_DIR}")